# Near-fault ground motion across a fault: a SeisSol receiver tutorial

**What you will learn**

This notebook takes the output of 3-D dynamic-rupture earthquake simulations of a
San Andreas fault segment (run with the SeisSol code) and asks two classic questions
about strong ground motion:

1. **How fast does shaking decay as you move away from the fault?**  (Figure 1)
2. **Which *direction* of shaking dominates, and how does that change across the fault?**  (Figure 2)

You can compare **one, two, or three (or more) cases** at once — for example different
**material models** for the same earthquake:
- **CVM** — a realistic 3-D *community velocity model* (rock gets softer/slower near the surface).
- **constant** — a single uniform rock everywhere (a control case).
- a **third** case of your choosing.

Comparing them isolates the effect of the 3-D Earth structure on the ground motion
(*site amplification*).

> All equations below are written in plain ASCII (no rendered math) on purpose.


## The data we use: off-fault "receivers"

A **receiver** is a virtual seismometer. SeisSol writes a time series at each one.
For this study we placed receivers on the **free surface (z = 0)** along a straight line
that crosses the fault at a right angle — a **fault-normal profile** — at distances of
2, 5, 10, and 20 km on each side, plus one on the fault trace itself (the "epicenter").

Each receiver file `safs-receiver-NNNNN-*.dat` is a text table:
```
# header lines:  TITLE, VARIABLES, and the station coordinates x1 x2 x3 (UTM metres, z up)
# columns:       Time, xx, yy, zz, xy, yz, xz, v1, v2, v3
```
- `v1, v2, v3` are the ground **velocity** components in metres/second.
- `v1` points East (model x), `v2` points North (model y), `v3` points up (vertical).
- Sampling is fine (dt = 0.005 s = 200 Hz), so these are good for peak ground motion.

We only need the **velocity** columns (`v1, v2, v3`) and the station coordinates.


## 1. Mount Google Drive (where your receiver files live)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Imports

In [ ]:
import os, re, glob
import numpy as np
import matplotlib.pyplot as plt

## 3. Point to the cases you want to compare

`RUN_DIRS` maps a **label** (used in the legends) to the **folder** that holds that case's
`safs-receiver-*.dat` files. Add or remove entries freely — **the notebook plots whichever
folders actually exist**, so it works with one, two, or three (or more) cases. You only need
the receiver `.dat` files uploaded to Drive, not the whole output.

In [ ]:
# EDIT this dict: label -> Drive folder. Comment out or add lines as needed.
RUN_DIRS = {
    "CVM":      "/content/drive/MyDrive/seisol_quakeworx/output_safs_v2.2.0_cvm_mat",
    "constant": "/content/drive/MyDrive/seisol_quakeworx/output_safs_v2.2.0_constant_mat_onfaultpoints",
    # "case 3":  "/content/drive/MyDrive/seisol_quakeworx/output_safs_v2.2.0_other",   # <- add a 3rd here
}

for name, d in RUN_DIRS.items():
    n = len(glob.glob(os.path.join(d, "safs-receiver-*.dat"))) if os.path.isdir(d) else 0
    print(f"{name:10s}: {'found' if n else 'MISSING'}  ({n} receiver files)  {d}")

## 4a. Where the profile stations are

The fault-normal profile receivers sit at the (x, y) positions below. The last number is
the **signed offset**: negative = SW side of the fault, positive = NE side, 0 = on the trace.
These come from how the stations were generated (committed in `safs_receivers.dat`).

In [ ]:
# label -> (x_UTM, y_UTM, signed fault-normal offset in km)
PROFILE = {
    "epicenter":            (604801.2000, 3705030.0000,   0.0),
    "fault_normal_SW_20km": (590745.5748, 3690801.8799, -20.0),
    "fault_normal_SW_10km": (597773.3874, 3697915.9400, -10.0),
    "fault_normal_SW_5km":  (601287.2937, 3701472.9700,  -5.0),
    "fault_normal_SW_2km":  (603395.6375, 3703607.1880,  -2.0),
    "fault_normal_NE_2km":  (606206.7625, 3706452.8120,   2.0),
    "fault_normal_NE_5km":  (608315.1063, 3708587.0300,   5.0),
    "fault_normal_NE_10km": (611829.0126, 3712144.0600,  10.0),
    "fault_normal_NE_20km": (618856.8252, 3719258.1201,  20.0),
}
print(f"{len(PROFILE)} profile stations (epicenter + 4 SW + 4 NE)")

## 4b. The "fault frame" (why we rotate the velocity)

The fault here is a strike-slip fault that runs roughly NW-SE, so it is **not** aligned with
the model x/y axes. The raw `v1` (East) and `v2` (North) therefore MIX two physically
different motions. We rotate the horizontal velocity into a **fault frame**:

```
n  = fault-NORMAL direction   (perpendicular to the fault, pointing NE)
t  = fault-PARALLEL direction (along strike)
```

The profile was built by stepping along `n`, so we read `n` straight off the geometry:

```
n = (position_NE_20km - position_SW_20km) / |position_NE_20km - position_SW_20km|
t = (n_y, -n_x)            # n rotated by -90 degrees in the map plane
```

Then the three meaningful velocity components at each time step are:

```
v_parallel = v1*t_x + v2*t_y      # along the fault (Love/SH-type motion)
v_normal   = v1*n_x + v2*n_y      # across the fault (carries the directivity pulse)
v_vertical = v3                   # up-down
```


In [ ]:
p_NE = np.array(PROFILE["fault_normal_NE_20km"][:2])
p_SW = np.array(PROFILE["fault_normal_SW_20km"][:2])
N = (p_NE - p_SW) / np.linalg.norm(p_NE - p_SW)   # fault-normal unit vector (points NE)
T = np.array([N[1], -N[0]])                        # fault-parallel (strike) unit vector
print("fault-normal  n =", N.round(3))
print("fault-parallel t =", T.round(3))

## 4c. Map of the experiment (station layout)

Before looking at any data, let's see **where the stations are**. We draw:
- the **fault** as a straight line through the epicenter along the strike direction `t`,
- the **profile stations** as numbered dots along the fault-normal direction `n`,
- the **NE side** (red, offset > 0) and **SW side** (blue, offset < 0).

Coordinates are shown **relative to the epicenter, in km**, so the picture is easy to read.
The numbers next to each dot are **station IDs** — the printed key below maps each ID to its
distance from the fault, and you can refer back to them when reading Figures 1 and 2.

In [ ]:
epi = np.array(PROFILE["epicenter"][:2])
ordered = sorted(PROFILE.items(), key=lambda kv: kv[1][2])      # SW(-) ... NE(+)
station_no = {lab: i + 1 for i, (lab, _) in enumerate(ordered)} # 1..9 along the profile

fig, ax = plt.subplots(figsize=(7.5, 7.5))

# fault: straight line through the epicenter along strike t, +/- 25 km
L = 25.0
ax.plot([-L*T[0], L*T[0]], [-L*T[1], L*T[1]], color="0.3", lw=3,
        solid_capstyle="round", zorder=1, label="fault (local strike)")

# faint line linking the stations (the profile direction n)
pts = {lab: ((x-epi[0])/1000, (y-epi[1])/1000) for lab,(x,y,_) in PROFILE.items()}
line = np.array([pts[lab] for lab,_ in ordered])
ax.plot(line[:,0], line[:,1], "--", color="0.75", lw=1, zorder=1)

# numbered stations, coloured by side
for lab,(x,y,off) in PROFILE.items():
    dx, dy = pts[lab]
    c = "C3" if off > 0 else ("C0" if off < 0 else "k")     # NE red, SW blue, epicenter black
    ax.scatter(dx, dy, s=95, color=c, edgecolor="white", linewidth=1.2, zorder=3)
    ax.annotate(str(station_no[lab]), (dx, dy), textcoords="offset points",
                xytext=(7, 6), fontsize=11, fontweight="bold", color=c, zorder=4)

# NE / SW side labels, placed out along +/- the fault-normal n
ax.text( N[0]*23,  N[1]*23, "NE side", color="C3", fontsize=13, fontweight="bold", ha="center")
ax.text(-N[0]*23, -N[1]*23, "SW side", color="C0", fontsize=13, fontweight="bold", ha="center")

# legend proxies
ax.scatter([], [], color="C3", label="NE-side station (offset > 0)")
ax.scatter([], [], color="C0", label="SW-side station (offset < 0)")
ax.scatter([], [], color="k",  label="on-fault (epicenter)")

ax.set_aspect("equal")
ax.set_xlabel("East of epicenter  [km]"); ax.set_ylabel("North of epicenter  [km]")
ax.set_title("Map of the fault-normal station profile\n(numbers = station IDs)")
ax.grid(True, alpha=0.3); ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

print("Station key (ID -> location):")
for lab, _ in ordered:
    off = PROFILE[lab][2]
    side = "NE" if off > 0 else ("SW" if off < 0 else "on-fault")
    print(f"  {station_no[lab]:>2d}   {side:8s}  |distance| {abs(off):4.0f} km    ({lab})")

## 5. Read each receiver and reduce it to peak values

For each profile station we:
1. read its coordinates from the header and match it to the `PROFILE` table,
2. load the velocity time series `v1, v2, v3`,
3. rotate into the fault frame (`v_parallel, v_normal, v_vertical`),
4. compute the **peak** of each component and the overall **PGV** (peak ground velocity).

```
PGV_3D        = max_t  sqrt(v_parallel^2 + v_normal^2 + v_vertical^2)   # the headline scalar
peak_parallel = max_t  |v_parallel|
peak_normal   = max_t  |v_normal|
peak_vertical = max_t  |v_vertical|
```
We then assign each loaded case a colour/marker automatically, so the figures work for
**any number of cases**.

In [ ]:
def read_station_coords(path):
    """Read x1,x2,x3 (UTM metres) from the receiver-file header."""
    xy = {}
    for line in open(path).readlines()[:8]:                 # coords sit in the first few '# x.' lines
        m = re.match(r"#\s*x([123])\s+([-+0-9.eE]+)", line)
        if m:
            xy[int(m.group(1))] = float(m.group(2))
    return np.array([xy[1], xy[2]])

def load_profile(run_dir):
    """Return {station_label: dict of peak values} for the profile stations in one run."""
    out = {}
    for f in sorted(glob.glob(os.path.join(run_dir, "safs-receiver-*.dat"))):
        xy = read_station_coords(f)
        label, dist = min(((lab, np.hypot(xy[0]-sx, xy[1]-sy))
                           for lab,(sx,sy,_) in PROFILE.items()), key=lambda p: p[1])
        if dist > 50.0:                       # >50 m from every profile point -> not on this profile
            continue
        data = np.loadtxt(f, comments="#", skiprows=2)      # skip TITLE + VARIABLES lines
        v = data[:, 7:10]                                   # columns v1, v2, v3
        v_par = v[:,0]*T[0] + v[:,1]*T[1]
        v_nor = v[:,0]*N[0] + v[:,1]*N[1]
        v_ver = v[:,2]
        out[label] = dict(
            offset        = PROFILE[label][2],
            pgv3d         = float(np.max(np.sqrt(v_par**2 + v_nor**2 + v_ver**2))),
            peak_parallel = float(np.max(np.abs(v_par))),
            peak_normal   = float(np.max(np.abs(v_nor))),
            peak_vertical = float(np.max(np.abs(v_ver))),
        )
    return out

# Load every case whose folder exists and actually contains profile stations
runs = {}
for name, d in RUN_DIRS.items():
    if os.path.isdir(d):
        prof = load_profile(d)
        if prof:
            runs[name] = prof
            print(f"{name}: loaded {len(prof)} profile stations")
        else:
            print(f"{name}: folder found but no profile receivers matched - skipped")
    else:
        print(f"{name}: folder missing - skipped")
assert runs, "No cases loaded - check RUN_DIRS paths."

# Automatic styling so 1, 2, 3+ cases all get distinct colour/marker
PALETTE = ["C3", "C0", "C2", "C1", "C4", "C5"]      # red, blue, green, orange, ...
MARKERS = ["o", "s", "^", "D", "v", "P"]
STYLE = {name: dict(color=PALETTE[i % len(PALETTE)], marker=MARKERS[i % len(MARKERS)])
         for i, name in enumerate(runs)}
print("\ncases to plot:", list(runs))

## Figure 1 — How ground motion decays away from the fault

We plot **PGV** against **distance from the fault** on **log-log axes** (a power-law decay
becomes a straight line). Each case gets its own colour; the **NE side is solid** and the
**SW side is dashed**, so you can spot cross-fault asymmetry.

Things to notice:
- Cases with softer near-surface rock (e.g. CVM) sit **higher** -> stronger shaking (site amplification).
- A gap between the **NE (solid)** and **SW (dashed)** lines of the same colour = real cross-fault asymmetry.
- The grey lines are **reference slopes** `R^-1` and `R^-1.5`; compare your decay against them.

(The on-fault station #5 is left out here: a log axis cannot show distance = 0, and it records
fault slip rather than radiated waves.)


In [ ]:
def sides(prof, key):
    """Split a quantity into (|offset|, value) for NE (offset>0) and SW (offset<0)."""
    ne = sorted((abs(d["offset"]), d[key]) for d in prof.values() if d["offset"] > 0)
    sw = sorted((abs(d["offset"]), d[key]) for d in prof.values() if d["offset"] < 0)
    arr = lambda L: (np.array([p[0] for p in L]), np.array([p[1] for p in L]))
    return arr(ne), arr(sw)

fig, ax = plt.subplots(figsize=(7, 5.5))
for name, prof in runs.items():
    ne, sw = sides(prof, "pgv3d")
    ax.loglog(ne[0], ne[1], "-",  **STYLE[name], label=f"{name}  (NE side)")
    ax.loglog(sw[0], sw[1], "--", color=STYLE[name]["color"], marker=STYLE[name]["marker"],
              mfc="none", label=f"{name}  (SW side)")

# reference geometric-spreading slopes, anchored at the first case's nearest NE point
first = next(iter(runs.values()))
ne0, _ = sides(first, "pgv3d")
if len(ne0[0]):
    x0, y0 = ne0[0][0], ne0[1][0]
    rr = np.array([x0, 20.0])
    for power, ls in [(-1.0, ":"), (-1.5, "--")]:
        ax.loglog(rr, y0*(rr/x0)**power, ls, color="0.5", lw=1.2, label=f"reference R^{power:.1f}")

ax.set_xlabel("distance from the fault  [km]")
ax.set_ylabel("PGV  (peak ground velocity)  [m/s]")
ax.set_title("Figure 1 - ground-motion decay with distance from the fault")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Figure 2 — Which direction of shaking dominates?

The same peak velocities, **split into the three fault-frame components**
(fault-parallel, fault-normal, vertical) and plotted against the **signed** offset
(SW on the left, NE on the right). One panel per component; one line per case.

Things to notice:
- For a **strike-slip** fault, **fault-parallel** motion is usually largest near the fault.
- The **fault-normal** component carries the forward-**directivity** pulse - watch for
  left/right asymmetry.
- If one case stays **above** the others in every panel, its shaking is amplified broadband.

We drop the on-fault station #5 (offset 0): its huge fault-slip amplitude would squash the
scale and hide the off-fault trend.


In [ ]:
def signed(prof, key):
    """Return (signed_offset, value) for off-trace stations, sorted left-to-right."""
    pts = sorted((d["offset"], d[key]) for d in prof.values() if d["offset"] != 0.0)
    return np.array([p[0] for p in pts]), np.array([p[1] for p in pts])

components = [("peak_parallel", "fault-parallel"),
             ("peak_normal",   "fault-normal"),
             ("peak_vertical", "vertical")]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
for ax, (key, title) in zip(axes, components):
    for name, prof in runs.items():
        x, y = signed(prof, key)
        ax.plot(x, y, "-", **STYLE[name], label=name)
    ax.axvline(0, color="k", lw=0.8, ls=":")                 # the fault location
    ax.set_xlabel("offset from fault  [km]\n(SW < 0 , NE > 0)")
    ax.set_title(f"peak {title} velocity")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("peak velocity  [m/s]")
axes[0].legend(fontsize=9)
fig.suptitle("Figure 2 - peak velocity by component across the fault")
plt.tight_layout(); plt.show()

## What to take away

- **Distance matters a lot.** PGV drops quickly within the first few km of the fault
  (Figure 1). Compare your decay to the grey `R^-1` line: steeper means energy falls off
  faster than simple spreading.
- **The 3-D Earth structure amplifies shaking.** Cases with a realistic velocity model sit
  above the uniform control - that gap is the *site effect*.
- **Strike-slip earthquakes shake mostly parallel to the fault near it**, but the
  **fault-normal** directivity pulse and its left/right asymmetry are what make near-fault
  ground motion dangerous and hard to predict (Figure 2).

### Try it yourself
- Add a third case to `RUN_DIRS` (cell 3) and re-run - every figure updates automatically.
- In Figure 1, is the decay closer to `R^-1` or `R^-1.5`? Is one side of the fault stronger?
- In Figure 2, which component is largest at station #4 (2 km)? At #1/#9 (20 km)?
